# CBC additional robustness analysis — RUN ALL once (V4 Colab-safe)

> **V4 Colab compatibility fix:** this notebook pins the Python↔R bridge used by the archived code,\n> forces `RPY2_CFFI_MODE=ABI`, and performs a `numpy2ri` preflight before any long analysis begins.\n> If an older notebook has already failed after importing `rpy2`, use **Runtime → Disconnect and delete runtime**\n> (or open this notebook in a fresh runtime) before pressing **Run all**.\n\nThis notebook **does not overwrite or modify the reference NIBFS running/results**. It reads the two input ZIPs, creates a separate additional analysis workspace, and runs only analyses that were still missing:

1. **GSE15852 paired-bootstrap correction** from the already fixed prediction probabilities — no model refit.
2. **Repeated strict training-fold-fitted analysis, 5×5** — repeat 1 is reused from the reference archive; only repeats 2–5 are newly feature-selected.
3. **Empirical stability-selection comparator** under the the same 10 repeated five-fold evaluation partitions — the comparison uses the corresponding NIBFS/DEG/mRMR/LASSO repeated results from the same evaluation partitions.
4. **Submission-ready consolidation** — updated external table, existing TCGA pair-bootstrap CI, metadata identity audit, SHA256 manifest, and a result-aware manuscript update guide.

### Before running
Put these items anywhere under **MyDrive** (the notebook auto-finds them):
- `NIBFS_Reproducibility_Archive_v1.1.0_REVIEW_ONLY(1).zip` (or the same v1.1.0 review archive under a slightly different filename)
- `tables-20260829T114033Z-1-001.zip`
- `CBC_ADDITIONAL ANALYSIS_RUN_ALL_20260829_V4.zip` (the notebook can auto-extract it; no manual folder extraction is required)

Then choose **Runtime → Run all**. The process is checkpointed; rerunning this notebook resumes completed folds rather than replacing the old reference results.


In [ ]:
# 1) Mount Drive and locate/extract the additional analysis package + input ZIPs
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import sys, zipfile, shutil

MYDRIVE = Path('/content/drive/MyDrive')
CONTENT = Path('/content')

def newest(paths):
    paths = [Path(p) for p in paths if Path(p).exists()]
    return max(paths, key=lambda p: p.stat().st_mtime) if paths else None

# A. First try to find an already-extracted package marker.
markers = list(MYDRIVE.rglob('CBC_ADDITIONAL ANALYSIS_RUN_ALL.marker')) + list(CONTENT.rglob('CBC_ADDITIONAL ANALYSIS_RUN_ALL.marker'))

# B. If the folder was not extracted, automatically find the package ZIP and extract it.
if not markers:
    package_zip_candidates = (
        list(MYDRIVE.rglob('CBC_ADDITIONAL ANALYSIS_RUN_ALL_20260829.zip')) +
        list(MYDRIVE.rglob('CBC_ADDITIONAL ANALYSIS_RUN_ALL_20260829_V4.zip')) +
        list(MYDRIVE.rglob('CBC_ADDITIONAL ANALYSIS_RUN_ALL_20260829_V3.zip')) +
        list(MYDRIVE.rglob('CBC_ADDITIONAL ANALYSIS_RUN_ALL_20260829_FIXED.zip')) +
        list(MYDRIVE.rglob('*CBC_ADDITIONAL ANALYSIS_RUN_ALL*20260829*.zip')) +
        list(CONTENT.glob('CBC_ADDITIONAL ANALYSIS_RUN_ALL_20260829.zip')) +
        list(CONTENT.glob('CBC_ADDITIONAL ANALYSIS_RUN_ALL_20260829_V4.zip')) +
        list(CONTENT.glob('CBC_ADDITIONAL ANALYSIS_RUN_ALL_20260829_V3.zip')) +
        list(CONTENT.glob('CBC_ADDITIONAL ANALYSIS_RUN_ALL_20260829_FIXED.zip')) +
        list(CONTENT.glob('*CBC_ADDITIONAL ANALYSIS_RUN_ALL*20260829*.zip'))
    )
    package_zip = newest(package_zip_candidates)
    if package_zip is None:
        raise FileNotFoundError(
            'CBC additional robustness analysis package was not found. Put CBC_ADDITIONAL ANALYSIS_RUN_ALL_20260829_V4.zip '
            '(or the original CBC_ADDITIONAL ANALYSIS_RUN_ALL_20260829.zip) anywhere in MyDrive, then Run all again.'
        )

    AUTO_EXTRACT_ROOT = CONTENT / 'cbc_additional analysis_package_auto'
    if AUTO_EXTRACT_ROOT.exists():
        shutil.rmtree(AUTO_EXTRACT_ROOT)
    AUTO_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

    print('Package folder not yet extracted.')
    print('Auto-extracting:', package_zip)
    with zipfile.ZipFile(package_zip, 'r') as z:
        z.extractall(AUTO_EXTRACT_ROOT)

    markers = list(AUTO_EXTRACT_ROOT.rglob('CBC_ADDITIONAL ANALYSIS_RUN_ALL.marker'))

if not markers:
    raise FileNotFoundError(
        'Package ZIP was found, but CBC_ADDITIONAL ANALYSIS_RUN_ALL.marker is missing inside it. '
        'Please use the FIXED package supplied with this notebook.'
    )

PACKAGE_DIR = newest(markers).parent.resolve()
if str(PACKAGE_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR))

def candidates_for(patterns):
    found=[]
    for root in [MYDRIVE, CONTENT]:
        for pat in patterns:
            found.extend(root.rglob(pat) if root == MYDRIVE else root.glob(pat))
    return [p for p in found if p.is_file()]

repo_candidates = candidates_for([
    'NIBFS_Reproducibility_Archive_v1.1.0_REVIEW_ONLY(1).zip',
    'NIBFS_Reproducibility_Archive_v1.1.0_REVIEW_ONLY.zip',
    '*NIBFS*Reproducibility*REVIEW_ONLY*.zip',
])
table_candidates = candidates_for([
    'tables-20260829T114033Z-1-001.zip',
    '*tables*20260829*.zip',
])

if not repo_candidates:
    raise FileNotFoundError(
        'Could not find NIBFS_Reproducibility_Archive_v1.1.0_REVIEW_ONLY*.zip. '
        'Put that ZIP anywhere under MyDrive, then Run all again.'
    )
if not table_candidates:
    raise FileNotFoundError(
        'Could not find tables-20260829T114033Z-1-001.zip. '
        'Put that ZIP anywhere under MyDrive, then Run all again.'
    )

REPO_ZIP = newest(repo_candidates).resolve()
TABLES_ZIP = newest(table_candidates).resolve()
WORKSPACE = (MYDRIVE / 'CBC_ADDITIONAL ANALYSIS_RESULTS_20260829').resolve()

print('Package :', PACKAGE_DIR)
print('Repo ZIP:', REPO_ZIP)
print('Tables  :', TABLES_ZIP)
print('Output  :', WORKSPACE)
print('\nReference inputs are read-only.')


In [ ]:
# 2) Install and VERIFY the exact Python↔R bridge needed by the archived NIBFS code
# IMPORTANT: the archived code expects rpy2.robjects.numpy2ri and ABI mode.
import os, shutil, subprocess, sys, importlib

os.environ["RPY2_CFFI_MODE"] = "ABI"

if shutil.which("Rscript") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "r-base-core", "r-base-dev"])

# Remove any partially/mixed rpy2 3.6 namespace install left by the current Colab image.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y",
     "rpy2", "rpy2-rinterface", "rpy2-robjects"],
    check=False, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "numpy>=1.26,<3", "pandas>=2.0,<3", "scipy>=1.11,<2",
    "scikit-learn>=1.4,<2", "statsmodels>=0.14", "lightgbm>=4,<5",
    "matplotlib>=3.8,<4", "pyyaml>=6", "neuroCombat==0.2.12",
    "rpy2[numpy]==3.6.6"
])
importlib.invalidate_caches()

# Install limma only if missing.
check = subprocess.run(
    ["Rscript", "-e", 'quit(status=ifelse(requireNamespace("limma", quietly=TRUE),0,1))'],
    capture_output=True,
)
if check.returncode != 0:
    r_code = """
options(repos = c(CRAN = 'https://cloud.r-project.org'))
if (!requireNamespace('BiocManager', quietly=TRUE)) install.packages('BiocManager', quiet=TRUE)
BiocManager::install('limma', ask=FALSE, update=FALSE, quiet=TRUE)
"""
    subprocess.check_call(["Rscript", "-e", r_code])

# HARD preflight: do not start the long analysis unless the exact bridge used
# by src/feature_selection.py works in this runtime.
import numpy as np
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from rpy2.robjects.conversion import localconverter

with localconverter(ro.default_converter + numpy2ri.converter):
    ro.globalenv["cbc_test_x"] = np.asarray([1.0, 2.0, 3.0], dtype=float)

_bridge_sum = float(ro.r("sum(cbc_test_x)")[0])
if abs(_bridge_sum - 6.0) > 1e-12:
    raise RuntimeError(f"rpy2/numpy2ri preflight failed: expected 6.0, got {_bridge_sum}")

_limma_version = str(ro.r('as.character(packageVersion("limma"))')[0])
print("Dependency setup: PASS")
print("RPY2_CFFI_MODE :", os.environ.get("RPY2_CFFI_MODE"))
print("rpy2 bridge    : numpy2ri PASS")
print("limma version  :", _limma_version)


In [ ]:
# 3) RUN ALL missing analyses — this is the only analysis call
SS_SUBSAMPLES = 50
SS_SCREEN_K = 1000

from src_additional.run_all import main

master = main(
    repo_zip=str(REPO_ZIP),
    tables_zip=str(TABLES_ZIP),
    workspace=str(WORKSPACE),
    ss_subsamples=SS_SUBSAMPLES,
    ss_screen_k=SS_SCREEN_K,
)

print('\nDONE.')
print('Open this folder in Drive:')
print(WORKSPACE / 'results_additional' / '99_submission_ready_summary')

## What to send back after the run

Please send the **single ZIP generated from `CBC_ADDITIONAL ANALYSIS_RESULTS_20260829/results_additional/`** (or the whole `results_additional` folder). The most important files will be under `99_submission_ready_summary/`:

- `CBC_ADDITIONAL ANALYSIS_MASTER_SUMMARY.json`
- `MANUSCRIPT_UPDATE_GUIDE.md`
- `UPDATED_Table_S5A_External_Summary.csv`
- `UPDATED_Main_Table6_external_rows.csv`
- `CBC_ADDITIONAL ANALYSIS_OUTPUT_MANIFEST_SHA256.csv`

The old primary/repeated/random-anchor/LOCO outputs remain untouched.